# AFML: run the existing adaptation pipeline on Colab

This notebook changes the compute backend, not the research protocol. It uses an explicitly exported project bundle and optional audited synthetic source bank. No account keys, local virtual environment, reference clones, or arbitrary run folders are uploaded.

1. Open this notebook in Google Colab. Select **Runtime → Change runtime type → GPU** if a free GPU is available.
2. On your laptop, export a bundle using the README command (`examples/export_colab_bundle.py`). Include the bank/partition/environment inputs to run experiments; omit all three for code-only device checks.
3. Run the cells below in order. A CUDA check must pass before a longer experiment.

Free GPU allocation/runtime duration are variable: [official Colab FAQ](https://research.google.com/colaboratory/faq.html). Download results before the temporary runtime disappears. CUDA is not considered verified merely because this notebook exists.


In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "Select an available GPU runtime, then rerun this cell."
print("GPU:", torch.cuda.get_device_name())


## Upload only your explicit project bundle

The export helper includes a fixed source/config/test allowlist. With all three artifact arguments, it also validates and includes the chosen complete synthetic bank and its matching model/environment partitions. Select that ZIP below. Extracting into a new directory preserves any earlier run.


In [ ]:
from google.colab import files
import hashlib, io, json, zipfile
from pathlib import Path
from uuid import uuid4

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one export_colab_bundle ZIP."
archive_bytes = next(iter(uploaded.values()))
workspace = Path.cwd() / ("afml_" + uuid4().hex[:8])
workspace.mkdir()
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for entry in archive.infolist():
        destination = (workspace / entry.filename).resolve()
        assert destination.is_relative_to(workspace.resolve()), "Unsafe ZIP member path"
    archive.extractall(workspace)
repo = workspace / "amortized-gnn-tta"
bundle = json.loads((repo / "colab_bundle.json").read_text())
assert bundle["protocol"] == "colab-project-bundle-v1"
for relative, expected in bundle["files_sha256"].items():
    path = (repo / relative).resolve()
    assert path.is_relative_to(repo.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, relative
print("Verified bundle files:", len(bundle["files_sha256"]), "at", repo)


## Install the project, keeping Colab's supplied PyTorch when compatible

This uses our existing project dependencies. It does not install the laptop's CPU wheel or its virtual environment. If dependency installation requires a runtime restart, restart and rerun the setup; do not ignore import/version errors.


In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo) + "[dev]"], check=True)
import torch_geometric
print("PyTorch", torch.__version__, "PyG", torch_geometric.__version__)
assert torch.cuda.is_available(), "GPU runtime is required for this notebook's CUDA checks."


## Check the software on CPU, then the actual GPU path

Tests run small correctness fixtures. The device command exercises GCN/GAT/GraphSAGE × head/LoRA × supervised/unlabeled fitting, saving deltas and verifying ordinary CPU-model reload predictions. These random-model smoke checks are not accuracy experiments. Stop and inspect any failure before launching a larger run.


In [ ]:
environment = {**os.environ, "OMP_NUM_THREADS": "1"}
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=repo, env=environment, check=True)
smoke = repo / "runs" / "colab-cuda-smoke"
subprocess.run([sys.executable, "examples/check_adaptation_device.py", "--device", "cuda",
                "--output-dir", str(smoke)], cwd=repo, env=environment, check=True)
device_report = json.loads((smoke / "device_report.json").read_text())
assert device_report["status"] == "complete" and device_report["device"] == "cuda"
print("Actual CUDA checks:", len(device_report["checks"]), "GPU:", device_report["gpu"])


## Run the same development experiment

This requires a bundle exported with all three artifact arguments. One source model per meta-training label subset is selected; no held-out models/environments are used. The default below covers all three meta-training lineages, eleven shifts plus control at severity 0.5, both update methods and all three objectives. The protocol is identical to the local 40-step/15-patience development sweep. Set `all_meta_lineages=False` for a smaller first execution; keep its different coverage explicit.

Bank validation regenerates its source and checks actual fingerprints/scores. If the cloud software stack cannot reproduce those identities, the run fails visibly; do not bypass the check or mix a newly generated bank with an old model partition. Use a separately validated fresh bank instead. Exact cross-device numerical equality is not promised.


In [ ]:
assert all(bundle.get(key) for key in ("bank", "model_partition", "environments")), (
    "This is a code-only bundle. Export again with --bank, --model-partition and --environments.")
all_meta_lineages = True
review = repo / "runs" / ("colab-peft-" + uuid4().hex[:8])
command = [sys.executable, "examples/run_adaptation_review.py",
           "--bank", str(repo / bundle["bank"]),
           "--model-partition", str(repo / bundle["model_partition"]),
           "--environments", str(repo / bundle["environments"]),
           "--output-dir", str(review), "--device", "cuda", "--steps", "40", "--patience", "15"]
if all_meta_lineages:
    command.append("--all-meta-lineages")
subprocess.run(command, cwd=repo, env=environment, check=True)
report = json.loads((review / "review.json").read_text())
assert report["status"] == "complete"
print("Baselines:", len(report["baselines"]), "Adaptations:", len(report["adaptations"]))
for row in report["summary"]:
    print(row["method"], row["objective"], row["family"],
          f'{100*row["mean_before_accuracy"]:.2f}% → {100*row["mean_after_accuracy"]:.2f}%')


## Download the real weight deltas and provenance

The archive includes initial/adapted/delta tensors, factors, per-job metrics, the frozen plan, implementation snapshot, aggregate review and the CUDA smoke report. Keep it outside Git. Download time/storage depends on selected coverage. Colab's temporary disk is not persistent storage, and interrupted sweeps are not automatically resumable.


In [ ]:
import shutil
shutil.copy2(smoke / "device_report.json", review / "cuda_device_report.json")
result_zip = shutil.make_archive(str(workspace / "afml-colab-results"), "zip",
                                 root_dir=review.parent, base_dir=review.name)
files.download(result_zip)
